In [5]:
# !pip install -U onnxscript onnx onnxruntime

In [6]:
# Basic Imports
import torch
import numpy as np
import pandas as pd
import os
import torch.nn as nn

WINDOW_SIZE = 100

In [7]:
class TinyGestureNet(nn.Module):
    def __init__(self, num_classes=3):
        super(TinyGestureNet, self).__init__()

        # Block 1: Increased channels to 32, added BatchNorm
        self.conv1 = nn.Conv1d(in_channels=6, out_channels=32, kernel_size=5, stride=2, padding=2)
        self.bn1 = nn.BatchNorm1d(32) # Will be folded into conv1 during TFLite conversion
        self.relu1 = nn.ReLU()

        # Block 2: Depthwise Separable Convolution (Wider)
        self.depthwise = nn.Conv1d(in_channels=32, out_channels=32, kernel_size=5, padding=2, groups=32)
        self.pointwise = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=1)
        self.bn2 = nn.BatchNorm1d(64) # Will be folded into pointwise
        self.relu2 = nn.ReLU()

        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool1d(1)

        # Added dropout to prevent overfitting to specific phone orientations
        self.dropout = nn.Dropout(0.3)

        # Final classifier
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.depthwise(x)
        x = self.relu2(self.bn2(self.pointwise(x)))
        x = self.pool(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

In [8]:
ONNX_PATH = "tiny_gesture.onnx"
DATASET_BIN = "representative_dataset.bin"

# 1. Load the trained PyTorch Model
model = TinyGestureNet(num_classes=3)

if not os.path.exists("tiny_gesture_net.pth"):
    print("Error: tiny_gesture_net.pth not found. Train the model first.")
    exit(1)

model.load_state_dict(torch.load("tiny_gesture_net.pth"))
model.eval()

TinyGestureNet(
  (conv1): Conv1d(6, 32, kernel_size=(5,), stride=(2,), padding=(2,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU()
  (depthwise): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), groups=32)
  (pointwise): Conv1d(32, 64, kernel_size=(1,), stride=(1,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (gap): AdaptiveAvgPool1d(output_size=1)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=64, out_features=3, bias=True)
)

In [9]:
# 2. Export to ONNX (The universal bridge to TFLite / C++)
print("Exporting PyTorch model to ONNX...")
dummy_input = torch.randn(1, 6, WINDOW_SIZE)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=18,
    input_names=['input'],
    output_names=['output']
)
print(f"Success! Model topology saved to {ONNX_PATH}")

Exporting PyTorch model to ONNX...
[torch.onnx] Obtain model graph for `TinyGestureNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TinyGestureNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Success! Model topology saved to tiny_gesture.onnx


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [11]:
# 3. Generate the Representative Dataset (Crucial for INT8 Calibration)
print("Generating Representative Dataset for Activation Scaling...")

# Load a small chunk of our real data to calibrate the scales
df = pd.read_csv("gesture_dataset_raw.csv")
features = df[['ax', 'ay', 'az', 'gx', 'gy', 'gz']].values[:WINDOW_SIZE * 50] # 50 windows of data

# Apply the exact same scaling used in training
scaler_mean = np.load("scaler_mean.npy")
scaler_scale = np.load("scaler_scale.npy")
features = (features - scaler_mean) / scaler_scale

Generating Representative Dataset for Activation Scaling...


In [12]:
# Save as a raw float32 binary.
# Edge compilers (like TFLite or TVM) will feed this through the ONNX model
# to calculate the Zero-Point and Scale factors for INT8 conversion.
features.astype(np.float32).tofile(DATASET_BIN)
print(f"Calibration data saved to {DATASET_BIN}")
print("\nNext step: The model is ready to be compiled to INT8!")

Calibration data saved to representative_dataset.bin

Next step: The model is ready to be compiled to INT8!
